In [ ]:
import os
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from pickle import load
from transformers import AutoTokenizer, RobertaTokenizer
from sklearn.metrics import confusion_matrix
from transformers import RobertaForSequenceClassification, AlbertForSequenceClassification, DistilBertForSequenceClassification
from sklearn.metrics import classification_report
from typing import List
import sys
sys.path.append('../..')
from data.datasets import get_data_loaders_with_chatgpt_annotated_data

In [ ]:
DEFAULT_CONFIG = {
    # data details
    "data_dir": "../../../datasets", 
    "dataset_splits": [0.7, 0.9],

    # model details
    "model_name": "distilbert-base-uncased",
    "lr": 5e-5,
    "epochs": 30, 
    "patience": 10,
    "num_warmup_steps": 500,
    "batch_sizes": {
        "train": 16,
        "val": 64,
        "test": 64
    },
    # other details
    "device": 'cpu' if torch.cuda.is_available() else 'cpu',
    "initial_seed": 1,
    "num_seeds": 5,
    "checkpoints_dir": "training/results/checkpoints",
}
print("Using config:", DEFAULT_CONFIG)

In [3]:
MODELS_LIB = {
    "albert-base-v2": AlbertForSequenceClassification,
    "distilbert-base-uncased": DistilBertForSequenceClassification,
    "roberta-base": RobertaForSequenceClassification
}

# Load Data

In [ ]:
data_loaders, label_encoder, _  = get_data_loaders_with_chatgpt_annotated_data(DEFAULT_CONFIG, debug=False, inference=True)
data_loaders["test"].dataset.data.head()
# delete training folder in current directory
os.system("rm -rf training")


# Load Models

In [5]:
def load_trained_model(model_name, path_to_model_files):
    model = MODELS_LIB[model_name].from_pretrained(path_to_model_files, local_files_only=True).to(DEFAULT_CONFIG["device"])
    path_to_tokenizer = "/".join(path_to_model_files.split("/")[:-1])
    if model_name == "roberta-base":
        tokenizer = RobertaTokenizer.from_pretrained(path_to_tokenizer, local_files_only=True, use_fast=False)
    else:
        tokenizer = AutoTokenizer.from_pretrained(path_to_tokenizer, local_files_only=True)
    le = load(open(os.path.join(path_to_tokenizer,'label_encoder.pkl'), 'rb'))
    model.eval()
    return model, tokenizer, le

In [1]:
def test_models(model_name:str, models):
    predictions = {seed: [] for seed in models[model_name]}
    for seed in models[model_name]:
        model, _, le = load_trained_model(
            model_name,
            seed
        )   
        for batch in tqdm(data_loaders["train"]):
            batch = {k: v.to(DEFAULT_CONFIG["device"]) for k, v in batch.items()}
            with torch.no_grad():
                output = model(**batch)
                batch_predictions = torch.argmax(output.logits, dim=-1)
            for idx, pred in enumerate(batch_predictions.tolist()):
                predictions[seed].append({"y_hat_enc": pred, "y_enc": batch["labels"].flatten().tolist()[idx], })
        predictions[seed] = pd.DataFrame(predictions[seed])
        predictions[seed]["y_hat"] = le.inverse_transform(predictions[seed]["y_hat_enc"])
        predictions[seed]["y"] = le.inverse_transform(predictions[seed]["y_enc"])
    return predictions

In [7]:
ONE_SEED_ONLY = False

In [ ]:
models = {}
checkpoints = [f for f in os.listdir('.') if os.path.isdir(f)]
for checkpoint in checkpoints:
    model_name = checkpoint.split(' ')[1][9:]
    seeds = [
        os.path.join('.', checkpoint, f)
        for f in os.listdir(os.path.join('.', checkpoint))
        if os.path.isdir(os.path.join('.', checkpoint, f))
    ]
    models[model_name] = seeds if not ONE_SEED_ONLY else seeds[:1]
models

In [9]:
def get_results(model, df):
    performance = classification_report(df["y"],
                                        df["y_hat"],
                                        target_names=df["y"].unique(),
                                        zero_division=0.0,
                                        output_dict=True)
    performance = pd.DataFrame(performance).T.sort_values(by="f1-score",
                                                          ascending=True)
    performance_to_plot = performance.drop(["accuracy", "macro avg", "weighted avg"])
    performance_to_plot[["f1-score"]].plot(kind="barh", figsize=(10, 20))
    # add numbers to the plot
    for i, v in enumerate(performance_to_plot["f1-score"]):
        plt.text(v, i, f"{v:.2f}", va="center")
    plt.title(f"F1 performance for {model}")
    plt.show()
    cm = confusion_matrix(df["y"], df["y_hat"], labels=df["y"].unique())
    plt.figure(figsize=(30, 30))
    sns.heatmap(cm,
                annot=True,
                fmt="d",
                xticklabels=df["y"].unique(),
                yticklabels=df["y"].unique())
    plt.title(f"Confusion matrix for {model}")
    plt.show()
    plt.figure(figsize=(10, 10))
    sns.set_theme(style="whitegrid")
    sns.scatterplot(data=performance_to_plot, x="f1-score", y="support")
    sns.scatterplot(data=performance_to_plot, x="recall", y="support")
    sns.scatterplot(data=performance_to_plot, x="precision", y="support")
    plt.legend(["f1-score", "recall", "precision"])
    plt.title(f"Performance metrics for {model}")
    plt.xlabel("Metric")
    plt.ylabel("Support")
    plt.show()
    return performance

# Distilbert

In [ ]:
predictions_distilbert = test_models("distilbert-base-uncased", models)
predictions_distilbert = pd.concat([predictions_distilbert[seed] for seed in predictions_distilbert], ignore_index=True)


In [ ]:
performance = get_results("distilbert-base-uncased", predictions_distilbert)

In [ ]:
# print the full performance table, its not showing all the rows when printed
pd.set_option('display.max_rows', None)
performance.sort_values(by="f1-score", ascending=False)